In [1]:
import pandas as pd
import numpy as np
import textwrap
import matplotlib.pyplot as plt
import math
import seaborn as sns
import datamol as dm
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from multiprocessing import Pool
import os
from pathlib import Path
import hashlib
from sklearn.model_selection import GroupShuffleSplit

from itertools import islice
df_well=pd.read_pickle('/home/ethan2/GrowthCurve/data/df_well_fingerprints.pkl')

In [2]:
import pandas as pd

# Keep only the two columns, drop NaNs, strip whitespace, drop exact duplicate pairs
pairs = (
    df_well[["Compound", "Smiles"]]
    .dropna()
    .assign(
        Compound=lambda d: d["Compound"].astype(str).str.strip(),
        Smiles=lambda d: d["Smiles"].astype(str).str.strip(),
    )
    .drop_duplicates()
)

# 1) Does every Compound map to exactly one SMILES?
smiles_per_compound = pairs.groupby("Compound")["Smiles"].nunique()
compounds_with_multiple_smiles = (
    pairs.groupby("Compound")["Smiles"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="smiles")
)
compounds_with_multiple_smiles["n_smiles"] = compounds_with_multiple_smiles["smiles"].str.len()
compounds_with_multiple_smiles = compounds_with_multiple_smiles.query("n_smiles > 1")

compound_to_smiles_is_unique = compounds_with_multiple_smiles.empty

# 2) Do different Compounds share the same SMILES?
compounds_per_smiles = pairs.groupby("Smiles")["Compound"].nunique()
smiles_used_by_multiple_compounds = (
    pairs.groupby("Smiles")["Compound"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="compounds")
)
smiles_used_by_multiple_compounds["n_compounds"] = smiles_used_by_multiple_compounds["compounds"].str.len()
smiles_used_by_multiple_compounds = smiles_used_by_multiple_compounds.query("n_compounds > 1")

smiles_to_compound_is_unique = smiles_used_by_multiple_compounds.empty

# 3) Summary
print(f"Unique mapping Compound→SMILES: {compound_to_smiles_is_unique}")
print(f"Unique mapping SMILES→Compound: {smiles_to_compound_is_unique}")

if not compound_to_smiles_is_unique:
    print("\nCompounds with >1 distinct SMILES:")
    print(compounds_with_multiple_smiles.sort_values("n_smiles", ascending=False).head(20))

if not smiles_to_compound_is_unique:
    print("\nSMILES used by >1 distinct Compound:")
    print(smiles_used_by_multiple_compounds.sort_values("n_compounds", ascending=False).head(20))


Unique mapping Compound→SMILES: False
Unique mapping SMILES→Compound: True

Compounds with >1 distinct SMILES:
          Compound                                             smiles  \
198  Ciprofloxacin  [C1CNCCN1c(c2)c(F)cc3c2N(C4CC4)C=C(C3=O)C(=O)O...   

     n_smiles  
198         2  


This is fine, won't be looking at positive controls anyway 

In [3]:
df_no_nan = df_well.dropna(subset=["maccs_fp", "ecfp_fp", "rdkit_fp"]).copy()

def fp_key(r):
    fp = np.concatenate([r["maccs_fp"], r["ecfp_fp"], r["rdkit_fp"]])
    # Hash the raw bytes; consistent within the dataset as long as dtype/shape are consistent
    return hashlib.sha1(fp.tobytes()).hexdigest()

df_no_nan["fp_key"] = df_no_nan.apply(fp_key, axis=1)

collisions = (
    df_no_nan.groupby("fp_key")["Compound"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="compounds")
)
collisions["n_compounds"] = collisions["compounds"].str.len()
collisions = collisions.query("n_compounds > 1")

if collisions.empty:
    print("✅ No duplicate concatenated fingerprints across different compounds.")
else:
    print("⚠️ Fingerprint collisions found:")
    print(collisions[["fp_key", "n_compounds", "compounds"]])


TARGET_CONC = 50
TARGET_TP   = 12.48

# Rows at the target (Timepoint, Concentration)
at_cell = df_well[
    np.isclose(df_well["Concentration"], TARGET_CONC)
    & np.isclose(df_well["Timepoint"], TARGET_TP)
].dropna(subset=["Compound", "is_Active"]).copy()

# For each compound, consider it "active at the cell" if ANY replicate is active
active_by_compound = (
    at_cell.groupby("Compound")["is_Active"]
           .max()                # any True/1 -> True
           .astype(bool)
)

def count_active(comp_list):
    # sum True for compounds present & active at the cell
    return int(sum(active_by_compound.get(c, False) for c in comp_list))

# Add the column to collisions
collisions["n_active_C50_T12.48"] = collisions["compounds"].apply(count_active)

⚠️ Fingerprint collisions found:
                                         fp_key  n_compounds  \
518    03ca64cfa8a528b5c12c4df9aa37beaaa0a1fe35            2   
758    05af9b879fa2548bcda815ace30094424b1b461f            2   
1364   09fd74b6e00786a948b799dcc61f7d1994d624ac            2   
1550   0b67db78e2d4a4e8b6fbcf760df6de62ab93ca98            2   
3464   199114a944f7cf3aa94c80623f4c8f9d9ddd0f4a            2   
...                                         ...          ...   
31606  e9d90a972beb85cfa57c43abf92122e53aae3520            2   
31896  ebee7015fc1038e342c9fc48b697890dda10dffb            2   
32337  ef5bf11086adc148b237ce12d8eb459c57ca3f4f            2   
33234  f5cee6bfb8f9e31436e8e4d160ff71e338910b6e            2   
34476  ff0aa0443ab7306378ce688f672f4c78691442eb            2   

                              compounds  
518               [Farnesol, Solanesol]  
758         [Florfenicol, UM0133378:01]  
1364   [Sulfadimethoxine, UM0118812:01]  
1550       [UM0119423:01, UM01

They have different SMILES, but identical fingerprints

In [4]:
maccs_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['maccs_fp'].values[0]
maccs_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['maccs_fp'].values[0]

ecfp_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['ecfp_fp'].values[0]
ecfp_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['ecfp_fp'].values[0]


rdkit_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['rdkit_fp'].values[0]
rdkit_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['rdkit_fp'].values[0]

print(np.array_equal(maccs_fp_Florfenicol, maccs_fp_UM0133378_01))
print(np.array_equal(ecfp_fp_Florfenicol, ecfp_fp_UM0133378_01))
print(np.array_equal(rdkit_fp_Florfenicol, rdkit_fp_UM0133378_01))

True
True
True


# Plot drawings

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from math import ceil
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

def _pick_rep_smiles(series: pd.Series) -> str:
    vc = series.value_counts()
    return vc.index[0]

def _slugify(s: str) -> str:
    """Safe-ish filename slug: lowercase, spaces->_, keep [a-z0-9._-], collapse repeats."""
    s = str(s).strip().lower().replace(" ", "_")
    s = re.sub(r"[^a-z0-9._-]+", "-", s)     # replace unsafe groups with '-'
    s = re.sub(r"-{2,}", "-", s).strip("-")  # collapse/trim dashes
    return s or "unknown"

def draw_collision_groups(
    collisions: pd.DataFrame,
    df_no_nan: pd.DataFrame,
    df_well: pd.DataFrame,
    outdir_base: str = "/home/ethan2/GrowthCurve/plots/collision_groups/drawings",
    molsPerRow: int = 3,
    subImgSize=(300, 300),
    target_conc: float = 50,
    target_tp: float = 12.48,
    name_maxlen: int = 22,
    per_name_slug_max: int = 40,       # max chars per compound in filename
    max_filename_chars: int = 220,     # hard cap for filename length (without extension)
):
    """
    Saves drawings under .../active or .../inactive. Filename includes slugified compound names.
    Active compound names are shown in red under each drawing.
    """
    os.makedirs(outdir_base, exist_ok=True)
    os.makedirs(os.path.join(outdir_base, "active"), exist_ok=True)
    os.makedirs(os.path.join(outdir_base, "inactive"), exist_ok=True)

    # Map compound -> active? at requested cell
    at_cell = df_well[
        np.isclose(df_well["Concentration"], target_conc)
        & np.isclose(df_well["Timepoint"], target_tp)
    ].dropna(subset=["Compound", "is_Active"]).copy()
    active_by_compound = (
        at_cell.groupby("Compound")["is_Active"].max().astype(bool)
        if not at_cell.empty else pd.Series(dtype=bool)
    )

    # PIL for colored text overlay
    try:
        from PIL import Image as PILImage
        from PIL import ImageDraw, ImageFont
    except Exception:
        PILImage = None
        ImageDraw = None
        ImageFont = None

    for ridx, row in collisions.iterrows():
        fp_key = row["fp_key"]
        n_active = row.get("n_active_C50_T12.48", None)
        folder = "active" if (n_active is not None and n_active > 0) else "inactive"
        outdir = os.path.join(outdir_base, folder)

        g = (
            df_no_nan.loc[df_no_nan["fp_key"] == fp_key, ["Compound", "Smiles"]]
            .dropna()
            .drop_duplicates()
        )
        if g.empty:
            print(f"[{ridx}] No rows found for fp_key={fp_key}")
            continue

        rep = g.groupby("Compound", as_index=True)["Smiles"].apply(_pick_rep_smiles)
        comp_order = list(rep.index)

        mols, act_flags = [], []
        for comp in comp_order:
            smi = rep.loc[comp]
            m = Chem.MolFromSmiles(smi)
            if m is None:
                print(f"[{ridx}] Skipping invalid SMILES for {comp}: {smi}")
                continue
            rdMolDraw2D.PrepareMolForDrawing(m)
            mols.append(m)
            act_flags.append(bool(active_by_compound.get(comp, False)))

        if not mols:
            print(f"[{ridx}] No valid molecules to draw for fp_key={fp_key}")
            continue

        # Legends (used only in SVG fallback)
        legends = []
        for comp, smi, is_act in zip(comp_order, rep.values, act_flags):
            tag = " (ACTIVE)" if is_act else ""
            legends.append(f"{comp}{tag}\n{smi}")

        # Build filename with compounds
        slugs = [ _slugify(c)[:per_name_slug_max] for c in comp_order ]
        comp_part = "__".join(slugs) if slugs else "no-compounds"
        prefix = f"collision_{ridx:03d}_n{len(mols)}__"
        base_name = prefix + comp_part
        if len(base_name) > max_filename_chars:
            # Trim compound part to fit, keep prefix intact
            keep = max(10, max_filename_chars - len(prefix) - 3)
            comp_part = comp_part[:keep] + "..."
            base_name = prefix + comp_part

        base = os.path.join(outdir, base_name)

        # Draw grid
        use_legends = None if PILImage is not None else legends
        img = Draw.MolsToGridImage(
            mols, molsPerRow=molsPerRow, subImgSize=subImgSize, legends=use_legends
        )

        if PILImage is not None and isinstance(img, PILImage.Image):
            # Overlay compound names: red if active, black otherwise
            draw = ImageDraw.Draw(img)
            try:
                font = ImageFont.load_default()
            except Exception:
                font = None

            cell_w, cell_h = subImgSize

            def short_name(s: str) -> str:
                s = str(s)
                return s if len(s) <= name_maxlen else s[:name_maxlen - 1] + "…"

            for i, comp in enumerate(comp_order):
                row_i = i // molsPerRow
                col_i = i % molsPerRow
                x_left = col_i * cell_w + 6
                y_text = row_i * cell_h + cell_h - 16
                color = (220, 0, 0) if act_flags[i] else (0, 0, 0)
                draw.text((x_left, y_text), short_name(comp), fill=color, font=font)

            out_png = base + ".png"
            img.save(out_png)
            print(f"Wrote {out_png} ({folder})")

        else:
            # Fallback: SVG with legends (active marked in text)
            svg_obj = Draw.MolsToGridImage(
                mols, molsPerRow=molsPerRow, subImgSize=subImgSize,
                legends=legends, useSVG=True
            )
            svg_str = svg_obj if isinstance(svg_obj, str) else getattr(svg_obj, "data", str(svg_obj))
            out_svg = base + ".svg"
            with open(out_svg, "w", encoding="utf-8") as f:
                f.write(svg_str)
            print(f"Wrote {out_svg} ({folder}) — PIL not available, active marked in text")

draw_collision_groups(
    collisions=collisions,
    df_no_nan=df_no_nan[df_no_nan['Control_Label'] == 0],
    df_well=df_well,
    outdir_base="/home/ethan2/GrowthCurve/plots/collision_groups/drawings",
    molsPerRow=3,
    subImgSize=(300, 300),
    target_conc=50,
    target_tp=12.48,
    name_maxlen=22,
    per_name_slug_max=40,
    max_filename_chars=220,
)


In [6]:
import os
import re
import math
import numpy as np
import pandas as pd
from io import BytesIO
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

def _pick_rep_smiles(series: pd.Series) -> str:
    vc = series.value_counts()
    return vc.index[0]

def _slugify(s: str) -> str:
    s = str(s).strip().lower().replace(" ", "_")
    s = re.sub(r"[^a-z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s or "unknown"

def draw_collision_groups_single_figure(
    collisions: pd.DataFrame,
    df_no_nan: pd.DataFrame,
    df_well: pd.DataFrame,
    out_png: str = "/home/ethan2/GrowthCurve/plots/collision_groups/collision_groups_all.png",
    molsPerRow: int = 3,
    subImgSize=(300, 300),
    target_conc: float = 50,
    target_tp: float = 12.48,
    name_maxlen: int = 22,
    groups_per_row: int = 2,
    group_pad: int = 24,
    title_height: int = 28,
    canvas_bg=(255, 255, 255),
    downscale: float = 1.0,
):
    # --- Pillow for composition ---
    try:
        from PIL import Image as PILImage
        from PIL import ImageDraw, ImageFont
    except Exception as e:
        raise RuntimeError("PIL is required for the single-figure composition.") from e

    os.makedirs(os.path.dirname(out_png), exist_ok=True)

    # Map compound -> active? at requested cell
    at_cell = df_well[
        np.isclose(df_well["Concentration"], target_conc)
        & np.isclose(df_well["Timepoint"], target_tp)
    ].dropna(subset=["Compound", "is_Active"]).copy()
    active_by_compound = (
        at_cell.groupby("Compound")["is_Active"].max().astype(bool)
        if not at_cell.empty else pd.Series(dtype=bool)
    )

    # Font for overlays
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    cell_w, cell_h = subImgSize

    def short_name(s: str) -> str:
        s = str(s)
        return s if len(s) <= name_maxlen else s[:name_maxlen - 1] + "…"

    def make_rdkit_grid_pil(mols, legends=None):
        """Create a per-group grid image reliably via Cairo → PNG bytes → PIL."""
        n = len(mols)
        if n == 0:
            return None
        rows = math.ceil(n / molsPerRow)
        width = molsPerRow * cell_w
        height = rows * cell_h

        # Prepare molecules for drawing (Kekulize, coords, etc.)
        for m in mols:
            rdMolDraw2D.PrepareMolForDrawing(m)

        d = rdMolDraw2D.MolDraw2DCairo(width, height, cell_w, cell_h)
        # Optional: tweak options if you like
        opts = d.drawOptions()
        # opts.legendFraction = 0.0  # we draw names ourselves, so no RDKit legends

        # RDKit will place mols in a grid automatically
        d.DrawMolecules(mols, legends=legends if legends is not None else [])
        d.FinishDrawing()
        png = d.GetDrawingText()

        return PILImage.open(BytesIO(png)).convert("RGB")

    def make_group_tile(ridx: int, fp_key, n_active_hint):
        """Build one collision-group tile with title + grid + red/black names."""
        g = (
            df_no_nan.loc[df_no_nan["fp_key"] == fp_key, ["Compound", "Smiles"]]
            .dropna()
            .drop_duplicates()
        )
        if g.empty:
            print(f"[{ridx}] No rows found for fp_key={fp_key}")
            return None

        rep = g.groupby("Compound", as_index=True)["Smiles"].apply(_pick_rep_smiles)
        comp_order = list(rep.index)

        mols, act_flags = [], []
        for comp in comp_order:
            smi = rep.loc[comp]
            m = Chem.MolFromSmiles(smi)
            if m is None:
                print(f"[{ridx}] Skipping invalid SMILES for {comp}: {smi}")
                continue
            mols.append(m)
            act_flags.append(bool(active_by_compound.get(comp, False)))

        if not mols:
            print(f"[{ridx}] No valid molecules to draw for fp_key={fp_key}")
            return None

        # Robust grid creation (avoids dtype object errors)
        img = make_rdkit_grid_pil(mols)

        # overlay names (bottom-left of each cell), red if active
        draw = ImageDraw.Draw(img)
        for i, comp in enumerate(comp_order[: len(mols)]):  # guard in case some SMILES were skipped
            row_i = i // molsPerRow
            col_i = i % molsPerRow
            x_left = col_i * cell_w + 6
            y_text = row_i * cell_h + cell_h - 16
            color = (220, 0, 0) if act_flags[i] else (0, 0, 0)
            draw.text((x_left, y_text), short_name(comp), fill=color, font=font)

        # title strip
        title = f"Group {ridx:03d} | fp_key={fp_key} | n_mols={len(mols)}"
        if n_active_hint is not None:
            title += f" | n_active@C{target_conc}_T{target_tp}={n_active_hint}"
        W = img.width
        H = img.height + title_height
        tile = PILImage.new("RGB", (W, H), canvas_bg)
        tdraw = ImageDraw.Draw(tile)
        tdraw.text((6, (title_height - 12) // 2), title, fill=(0, 0, 0), font=font)
        tile.paste(img, (0, title_height))
        tdraw.rectangle([(0, 0), (W - 1, H - 1)], outline=(220, 220, 220), width=1)
        return tile

    # Build all tiles
    tiles = []
    for ridx, row in collisions.iterrows():
        fp_key = row["fp_key"]
        n_active = row.get("n_active_C50_T12.48", None)
        tile = make_group_tile(ridx, fp_key, n_active)
        if tile is not None:
            tiles.append(tile)

    if not tiles:
        raise RuntimeError("No tiles were created; nothing to compose.")

    # Normalize sizes by padding (don’t rescale drawings)
    max_w = max(t.width for t in tiles)
    max_h = max(t.height for t in tiles)
    from PIL import Image as PILImage  # already imported; just for type hint clarity
    norm_tiles = []
    for t in tiles:
        if t.width != max_w or t.height != max_h:
            pad_w = max_w - t.width
            pad_h = max_h - t.height
            padded = PILImage.new("RGB", (max_w, max_h), canvas_bg)
            padded.paste(t, (pad_w // 2, pad_h // 2))
            norm_tiles.append(padded)
        else:
            norm_tiles.append(t)

    # Lay out group tiles on one big canvas
    n = len(norm_tiles)
    cols = max(1, int(groups_per_row))
    rows = math.ceil(n / cols)
    canvas_w = cols * max_w + (cols + 1) * group_pad
    canvas_h = rows * max_h + (rows + 1) * group_pad

    canvas = PILImage.new("RGB", (canvas_w, canvas_h), canvas_bg)
    for idx, tile in enumerate(norm_tiles):
        r = idx // cols
        c = idx % cols
        x = group_pad + c * (max_w + group_pad)
        y = group_pad + r * (max_h + group_pad)
        canvas.paste(tile, (x, y))

    if 0.0 < downscale < 1.0:
        new_size = (int(canvas.width * downscale), int(canvas.height * downscale))
        canvas = canvas.resize(new_size, resample=PILImage.Resampling.LANCZOS)

    canvas.save(out_png)
    print(f"Wrote single figure with {len(norm_tiles)} groups to: {out_png}")


# Example call mirroring your previous invocation:
draw_collision_groups_single_figure(
    collisions=collisions,
    df_no_nan=df_no_nan[df_no_nan['Control_Label'] == 0],
    df_well=df_well,
    out_png="/home/ethan2/GrowthCurve/plots/collision_groups/collision_groups_all.png",
    molsPerRow=3,
    subImgSize=(300, 300),
    target_conc=50,
    target_tp=12.48,
    name_maxlen=22,
    groups_per_row=4,     # try 2–3 depending on how many groups you have
    group_pad=24,
    title_height=28,
    downscale=1.0,        # e.g., 0.75 to shrink the final PNG
)


Wrote single figure with 79 groups to: /home/ethan2/GrowthCurve/plots/collision_groups/collision_groups_all.png


# Plot OD values

In [6]:
import os, re, math, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import islice

def _slugify(s: str) -> str:
    """Make a safe filename slug: lowercase, spaces->_, keep [a-z0-9._-], collapse repeats."""
    s = str(s).strip().lower().replace(" ", "_")
    s = re.sub(r"[^a-z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s or "unknown"

def _mean_abs_pairwise(vals: np.ndarray) -> float:
    x = np.asarray(vals, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        return np.nan
    diffs = np.abs(x[:, None] - x[None, :])
    upper = diffs[np.triu_indices(n, 1)]
    return float(upper.mean())

def save_od_diff_heatmaps_for_collisions(
    collisions: pd.DataFrame,
    df_well: pd.DataFrame,
    outdir: str = "/home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/",
    num_groups: int | None = None,
    agg: str = "mean",          # aggregate replicates per (Compound,T,C)
    fontsize_cap: int = 10,
    min_fontsize: int = 5,
    max_name_len: int = 16,
    vmin: float = 0.0,          # fixed legend min
    vmax: float = 0.2,          # fixed legend max
    title_width: int = 50,
    target_conc: float = 50.0,  # for active/inactive folder decision
    target_tp: float = 12.48,
    per_name_slug_max: int = 40,      # NEW: max chars per compound in filename
    max_filename_chars: int = 220,    # NEW: overall filename cap (without extension)
) -> None:
    """
    Heatmap:
      x-axis  = Timepoint (ascending)
      y-axis  = Concentration (descending so highest conc at BOTTOM)
    Color     = mean absolute pairwise ΔOD across compounds (fixed [vmin, vmax]).
    Text      = each compound’s aggregated OD; RED if active at that (T, C).
    Output    = .../active/ if any compound active at (target_tp, target_conc), else .../inactive/.
    Filenames = include slugified compound names.
    """
    need_cols = {"Compound", "Timepoint", "Concentration", "OD", "is_Active"}
    miss = need_cols - set(df_well.columns)
    if miss:
        raise ValueError(f"df_well missing required columns: {miss}")

    # Output dirs
    active_dir   = os.path.join(outdir, "active")
    inactive_dir = os.path.join(outdir, "inactive")
    os.makedirs(active_dir, exist_ok=True)
    os.makedirs(inactive_dir, exist_ok=True)

    rows_iter = collisions.itertuples(index=False)
    if num_groups is not None:
        rows_iter = islice(rows_iter, num_groups)

    for idx, row in enumerate(rows_iter):
        fp_key = getattr(row, "fp_key", f"group_{idx}")
        compounds = list(row.compounds) if hasattr(row, "compounds") else []
        if not compounds:
            print(f"[{idx}] Skipping: no compounds listed.")
            continue

        sub = df_well[df_well["Compound"].isin(compounds)].copy()
        if sub.empty:
            print(f"[{idx}] No matching rows in df_well for fp_key={fp_key}.")
            continue

        # Decide target folder: any compound active at (target_tp, target_conc)?
        at_cell = sub[
            np.isclose(sub["Concentration"], target_conc) &
            np.isclose(sub["Timepoint"],    target_tp)
        ]
        group_is_active = bool(at_cell["is_Active"].fillna(False).astype(bool).any())
        outdir_this = active_dir if group_is_active else inactive_dir

        # Aggregate OD + activity per (Compound, T, C)
        aggfunc = "median" if agg.lower() == "median" else "mean"
        per_comp = (
            sub.groupby(["Compound", "Timepoint", "Concentration"], as_index=False)
               .agg(OD=("OD", aggfunc), active=("is_Active", "max"))
        )

        # Build maps (T,C) -> {compound -> value}
        cell_od:  dict[tuple, dict] = {}
        cell_act: dict[tuple, dict] = {}
        for r in per_comp.itertuples(index=False):
            key = (getattr(r, "Timepoint"), getattr(r, "Concentration"))
            comp = getattr(r, "Compound")
            cell_od.setdefault(key, {})[comp]  = float(getattr(r, "OD"))
            cell_act.setdefault(key, {})[comp] = bool(getattr(r, "active"))

        # Axis sets: x=timepoints ascending, y=concs DESCENDING (highest at bottom)
        tp_vals  = sorted(per_comp["Timepoint"].unique())               # x-axis
        conc_vals_desc = sorted(per_comp["Concentration"].unique(), reverse=True)  # y-axis

        # Matrix of mean absolute pairwise ΔOD with rows=conc (desc), cols=timepoint (asc)
        diff_mat = pd.DataFrame(index=conc_vals_desc, columns=tp_vals, dtype=float)
        for c in conc_vals_desc:
            for t in tp_vals:
                od_map = cell_od.get((t, c), {})
                diff_mat.loc[c, t] = (
                    np.nan if not od_map else _mean_abs_pairwise(list(od_map.values()))
                )

        data = diff_mat.values.astype(float)
        finite = np.isfinite(data)
        if not finite.any():
            print(f"[{idx}] All cells are NaN for fp_key={fp_key}; skipping.")
            continue
        masked = np.ma.array(data, mask=~finite)

        # Figure size scales with grid size
        nY, nX = diff_mat.shape  # rows (conc), cols (time)
        fig_w = max(6, 0.9 * nX + 3)
        fig_h = max(6, 0.9 * nY + 3)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(masked, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Mean absolute pairwise ΔOD")

        # Axes labels/ticks
        ax.set_xticks(range(nX))
        ax.set_yticks(range(nY))
        ax.set_xticklabels([f"{t:g}" for t in diff_mat.columns], rotation=45, ha="right")
        ax.set_yticklabels([f"{c:g}" for c in diff_mat.index])
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Concentration")

        # Compounds order & title
        comp_order = sorted(set(per_comp["Compound"]), key=str)
        comp_wrapped = textwrap.fill(", ".join(map(str, comp_order)), width=title_width)
        ax.set_title(
            "ΔOD (mean |pairwise differences|) by (Timepoint, Concentration)\n"
            f"Compounds: {comp_wrapped}"
        )

        # --- Annotation visibility helpers ---
        def short(name: str, max_len: int) -> str:
            name = str(name)
            return name if len(name) <= max_len else name[:max_len-1] + "…"

        # Cell size in points
        cell_h_pt = fig_h * 72.0 / max(nY, 1)
        cell_w_pt = fig_w * 72.0 / max(nX, 1)
        lines_per_cell = max(1, len(comp_order))
        fs_auto = 0.9 * cell_h_pt / (lines_per_cell + 0.5)
        fs = max(min_fontsize, min(fontsize_cap, fs_auto))

        approx_char_w = 0.6 * fs
        max_chars_horiz = max(3, int(0.9 * cell_w_pt / approx_char_w) - 8)
        name_len = max(3, min(max_name_len, max_chars_horiz))

        # Draw per-line texts with colors (red if active at that (T, C), else black)
        for yi, c in enumerate(diff_mat.index):        # y = concentration (desc)
            for xi, t in enumerate(diff_mat.columns):  # x = timepoint (asc)
                od_map  = cell_od.get((t, c), {})
                act_map = cell_act.get((t, c), {})
                if not od_map:
                    continue

                L = len(comp_order)
                if L == 0:
                    continue
                dy = 0.8 / L
                start_y = yi - 0.4 + dy/2
                for j, comp in enumerate(comp_order):
                    v = od_map.get(comp, None)
                    active_here = bool(act_map.get(comp, False))
                    color = "red" if active_here else "black"
                    text = f"{short(comp, name_len)}: {'–' if v is None or (isinstance(v, float) and math.isnan(v)) else f'{v:.3f}'}"
                    ax.text(xi, start_y + j*dy, text,
                            ha="center", va="center", fontsize=fs, color=color, clip_on=True)

        fig.tight_layout()

        # ---------- Filename with compound names ----------
        short_fp = str(fp_key)[:10]
        slug_list = [_slugify(c)[:per_name_slug_max] for c in comp_order]
        comp_part = "__".join(slug_list) if slug_list else "no-compounds"
        prefix = f"ODHM_diff_{idx:03d}_{short_fp}__"
        base_name = prefix + comp_part
        if len(base_name) > max_filename_chars:
            keep = max(10, max_filename_chars - len(prefix) - 3)
            comp_part = comp_part[:keep] + "..."
            base_name = prefix + comp_part
        out_path = os.path.join(outdir_this, base_name + ".png")
        # ---------------------------------------------------

        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Wrote {out_path}  -> {'ACTIVE' if group_is_active else 'inactive'}")


save_od_diff_heatmaps_for_collisions(
    collisions=collisions,
    df_well=df_well[(df_well["Timepoint"] != 0) & (df_well["Control_Label"] != 1)],
    outdir="/home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/",
    num_groups=None,
    agg="mean",
    fontsize_cap=6,
    min_fontsize=6,
    max_name_len=7,
    vmin=0.0,
    vmax=0.1,
    title_width=50,
    target_conc=50.0,
    target_tp=12.48,
    per_name_slug_max=40,     # optional tuning
    max_filename_chars=220,   # optional tuning
)


Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_000_03ca64cfa8__farnesol__solanesol.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/active/ODHM_diff_001_05af9b879f__florfenicol__um0133378-01.png  -> ACTIVE
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_002_09fd74b6e0__sulfadimethoxine__um0118812-01.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_003_0b67db78e2__um0119423-01__um0133016-01.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_004_199114a944__z7592903406__z7670614062.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_005_1b5185ece2__z5060635472__z5367334035.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/inactive/ODHM_diff_006_1d398305e7__z3515200948__z3515200953.png  -> inactive
Wrote /home/ethan2/GrowthCurve/plots/coll

In [4]:
sorted(["your","deez",'nuts'])

['deez', 'nuts', 'your']